# 🏥 CareFlow — Notebook 7: SLA Compliance & Breach Analytics
## Clinical Regulatory Compliance, Risk Factors & Odds Ratio Analysis

**Author:** CareFlow Data Science Team  
**Standards:** CMS OP-18 (Emergency Department Length of Stay), Joint Commission Door-to-Doctor Standard  
**Dataset:** `careflow_analytics/output/mart_sla_compliance.parquet` (1,000 cases)

---

### Objectives:
1. Benchmark Door-to-Doctor ($<30$ min), Radiology Turnaround ($<45$ min), and Boarding ($<240$ min) compliance.
2. Conduct Chi-square tests of independence across clinical severity levels.
3. Fit Multivariable Logistic Regression models to identify independent predictors of SLA breach.
4. Calculate and visualize Odds Ratios (OR) with 95% Confidence Intervals (Forest Plot).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

plt.style.use('dark_background')
sns.set_palette('rocket')

sla_df = pd.read_parquet('../careflow_analytics/output/mart_sla_compliance.parquet')
print(f"Loaded {len(sla_df):,} patient SLA compliance records.")
sla_df.describe().T[['count', 'mean', 'std', 'min', '50%', 'max']]


In [ ]:
# 1. Door-to-Doctor Compliance Distribution & Violin Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram with SLA threshold line
sns.histplot(sla_df['door_to_doctor_minutes'].dropna(), kde=True, ax=axes[0], color='#06b6d4', bins=25)
axes[0].axvline(30, color='#f43f5e', linestyle='--', linewidth=2, label='SLA Target (30m)')
axes[0].set_title('Door-to-Doctor Wait Time Distribution')
axes[0].set_xlabel('Minutes')
axes[0].legend()

# Violin plot by Clinical Severity
sns.violinplot(data=sla_df, x='clinical_severity', y='door_to_doctor_minutes', order=['High', 'Medium', 'Low'],
               palette=['#f43f5e', '#f59e0b', '#10b981'], ax=axes[1])
axes[1].axhline(30, color='#f43f5e', linestyle='--', linewidth=2, label='Target (30m)')
axes[1].set_title('Wait Time by Triage Acuity Level')
axes[1].set_ylabel('Door-to-Doctor (Minutes)')
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# 2. Chi-Square Test of Independence: Severity vs SLA Breach
contingency_table = pd.crosstab(sla_df['clinical_severity'], sla_df['is_door_to_doctor_breached'])
chi2, p_val, dof, expected = stats.chi2_contingency(contingency_table)

print("Contingency Table (Severity vs Door-to-Doctor Breach):")
print(contingency_table)
print(f"\nChi-Square Statistic: {chi2:.4f}")
print(f"p-value:              {p_val:.5e} ({'Statistically Significant' if p_val < 0.05 else 'Not Significant'})")


In [ ]:
# 3. Multivariable Logistic Regression: Predicting Door-to-Doctor Breach
model_data = sla_df.dropna(subset=['door_to_doctor_minutes']).copy()
model_data['breach'] = model_data['is_door_to_doctor_breached'].astype(int)

logit_model = smf.logit("breach ~ C(clinical_severity, Treatment(reference='Low')) + C(patient_gender) + C(patient_age_group)", data=model_data).fit()
print(logit_model.summary())


In [ ]:
# 4. Odds Ratio Forest Plot
params = logit_model.params
conf = logit_model.conf_int()
conf['OR'] = params
conf.columns = ['Lower 95%', 'Upper 95%', 'OR']
odds_ratios = np.exp(conf).iloc[1:]  # Exclude intercept

plt.figure(figsize=(9, 5))
y_pos = range(len(odds_ratios))
plt.errorbar(odds_ratios['OR'], y_pos, 
             xerr=[odds_ratios['OR'] - odds_ratios['Lower 95%'], odds_ratios['Upper 95%'] - odds_ratios['OR']],
             fmt='o', color='#06b6d4', ecolor='#8b5cf6', elinewidth=2, capsize=4, markersize=7)

plt.axvline(1.0, color='#f43f5e', linestyle='--', alpha=0.7)
plt.yticks(y_pos, odds_ratios.index)
plt.xlabel('Odds Ratio (OR) with 95% Confidence Interval')
plt.title('Clinical Risk Factors for SLA Escalation (Logistic Regression)')
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()
